# Maske Tespiti

Bu projede yüz fotoğrafında maske var mı yok mu onu sınıflandıracağım.


In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')


### Data


In [ ]:
X=[]; y=[]
for lab,klasor in [(1,'data/with_mask'),(0,'data/without_mask')]:
    for f in os.listdir(klasor):
        p=os.path.join(klasor,f)
        im=cv2.imread(p,cv2.IMREAD_GRAYSCALE)
        if im is None: continue
        im=cv2.resize(im,(64,64))
        X.append(im.flatten()); y.append(lab)
X=np.array(X); y=np.array(y)
print(X.shape,np.bincount(y))


### EDA


In [ ]:
print('bos',np.isnan(X).sum())
plt.imshow(X[0].reshape(64,64),cmap='gray')
plt.title('ornek')
plt.axis('off')
plt.show()


### Görselleştirme


In [ ]:
plt.subplot(1,2,1); plt.imshow(X[y==1][0].reshape(64,64),cmap='gray'); plt.title('maskeli')
plt.subplot(1,2,2); plt.imshow(X[y==0][0].reshape(64,64),cmap='gray'); plt.title('maskesiz')
plt.show()


### Boş veri / scale


In [ ]:
from sklearn.preprocessing import MinMaxScaler
X=np.nan_to_num(X)
Xs=MinMaxScaler().fit_transform(X)


### Feature Engineering

Pikselleri düzleştirdim, ekstra ortalama parlaklık ekledim.


In [ ]:
parlak=Xs.mean(axis=1,keepdims=True)
Xs=np.hstack([Xs,parlak])


### Train Test Split


In [ ]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(Xs,y,test_size=0.25,random_state=42,stratify=y)


### 3 Model


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score,classification_report

for ad,m in [('LogReg',LogisticRegression(max_iter=200)),('KNN',KNeighborsClassifier()),('RF',RandomForestClassifier(random_state=42))]:
    m.fit(x_train,y_train)
    print(ad,accuracy_score(y_test,m.predict(x_test)))


In [ ]:
rf=RandomForestClassifier(random_state=42)
rf.fit(x_train,y_train)
print(classification_report(y_test,rf.predict(x_test)))


In [ ]:
import joblib
joblib.dump(rf,'../../models/cv_face_mask.joblib')


### Sonuç

Gerçek maskeli/maskesiz fotoğraflarla denedim. Set küçük olduğu için skor değişken olabilir ama pipeline çalışıyor. Hedefi temel olarak tutturdum.
